# ML Pipelines & Data Leakage Prevention - Assignment

Welcome to the assignment for ML Pipelines and Data Leakage Prevention.

Machine learning pipelines are essential tools for building robust, reproducible, and production-ready models. A pipeline chains together multiple processing steps, from data preprocessing to model training, ensuring that operations are applied consistently across training and inference. Properly designed pipelines prevent data leakage, simplify hyperparameter tuning, and make models easier to deploy and maintain.

Data leakage occurs when information from outside the training dataset is used to create the model, leading to overly optimistic performance estimates that fail in production. Leakage can be subtle: fitting a scaler on the entire dataset before splitting, using target-dependent features, or applying transformations that peek into the future. Understanding and preventing leakage is critical for building models that generalize to real-world data.

In this assignment, you will learn to construct pipelines that maintain proper data boundaries, handle heterogeneous data types, integrate with hyperparameter tuning, and prepare models for production deployment. You will also identify common leakage patterns and implement safeguards against them.

**What You Will Do in This Assignment**

* Build basic pipelines that chain preprocessing and modeling steps correctly.
* Use ColumnTransformer to handle mixed numeric and categorical data in a unified pipeline.
* Identify and fix data leakage scenarios in provided code examples.
* Integrate pipelines with GridSearchCV for safe hyperparameter tuning.
* Create production-ready pipelines that handle raw data end-to-end.
* Implement pipeline logic from scratch to understand fit/transform mechanics.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

1. [Introduction to ML Pipelines](#1)
2. [Basic Pipeline Construction](#2)
   - [Exercise 1: Build a Basic Pipeline](#ex-1)
3. [ColumnTransformer for Mixed Data](#3)
   - [Exercise 2: Implement ColumnTransformer](#ex-2)
4. [Data Leakage Identification](#4)
   - [Exercise 3: Find and Fix Leakage](#ex-3)
5. [Pipeline with Hyperparameter Tuning](#5)
   - [Exercise 4: Pipeline with GridSearchCV](#ex-4)
6. [Production Pipeline](#6)
   - [Exercise 5: End-to-End Production Pipeline](#ex-5)
7. [From-Scratch Implementation](#7)
   - [Exercise 6: Simple Pipeline from Scratch](#ex-6)
8. [Conclusion](#8)

<a name='1'></a>
## 1 - Introduction to ML Pipelines

**Pipelines** automate the workflow of transforming data and training models, ensuring consistency and preventing common errors.

### Why Use Pipelines?

1. **Prevent Data Leakage**: Ensure preprocessing fits only on training data
2. **Reproducibility**: Same transformations applied consistently
3. **Simplify Code**: Clean, readable workflow
4. **Enable Grid Search**: Tune preprocessing and model hyperparameters jointly
5. **Production Ready**: Easy serialization and deployment

### Pipeline Structure:

A pipeline is a sequence of steps:
$$\text{Pipeline} = [\text{Step}_1, \text{Step}_2, ..., \text{Step}_n, \text{Estimator}]$$

Each step (except the last) must implement:
- `fit(X, y)`: Learn parameters from training data
- `transform(X)`: Apply transformation to data

The final step (estimator) implements:
- `fit(X, y)`: Train the model
- `predict(X)`: Make predictions

### Pipeline Execution:

**Training** (`pipeline.fit(X_train, y_train)`):
```python
# Step 1: Fit and transform
X_temp = step1.fit(X_train, y_train).transform(X_train)
# Step 2: Fit and transform
X_temp = step2.fit(X_temp, y_train).transform(X_temp)
# Final: Fit estimator
estimator.fit(X_temp, y_train)
```

**Prediction** (`pipeline.predict(X_test)`):
```python
# Only transform (no fitting!)
X_temp = step1.transform(X_test)
X_temp = step2.transform(X_temp)
predictions = estimator.predict(X_temp)
```

### Data Leakage:

**Data leakage** occurs when information from test data influences training.

**Common Sources:**
1. **Preprocessing Leakage**: Fitting scaler on entire dataset
2. **Temporal Leakage**: Using future information to predict past
3. **Target Leakage**: Features derived from target
4. **Test Set Leakage**: Using test set for any training decisions

**Example of Leakage:**
```python
# WRONG: Fits scaler on ALL data including test
scaler = StandardScaler().fit(X)  # X includes test data!
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled)

# CORRECT: Fit only on training data
X_train, X_test = train_test_split(X)
scaler = StandardScaler().fit(X_train)  # Only training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

### Key Principles:

- **Fit only on training data**: Never fit transformers on test/validation sets
- **Transform consistently**: Use same fitted transformers for all data
- **Respect temporal order**: Don't use future to predict past
- **Validate carefully**: Use proper cross-validation with pipelines

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import unittests
import warnings
warnings.filterwarnings('ignore')

# Pipeline and preprocessing
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

# Models and evaluation
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Datasets
from sklearn.datasets import load_iris, load_breast_cancer, make_classification

# Serialization
import joblib
import pickle

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")

### Helper Functions

In [ ]:
def load_titanic_data():
    """Load Titanic dataset for mixed data type examples."""
    try:
        titanic = sns.load_dataset('titanic')
        # Select relevant columns
        titanic = titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
        titanic = titanic.dropna(subset=['embarked'])  # Drop few missing embarked values
        
        X = titanic.drop('survived', axis=1)
        y = titanic['survived']
        
        return X, y
    except:
        print("Titanic dataset not available, creating synthetic mixed data")
        # Create synthetic data with mixed types
        n_samples = 800
        df = pd.DataFrame({
            'numeric1': np.random.randn(n_samples),
            'numeric2': np.random.rand(n_samples) * 100,
            'category1': np.random.choice(['A', 'B', 'C'], n_samples),
            'category2': np.random.choice(['X', 'Y'], n_samples)
        })
        y = (df['numeric1'] + df['numeric2'] / 50 + 
             (df['category1'] == 'A').astype(int) * 2 + 
             np.random.randn(n_samples) * 0.5 > 1).astype(int)
        return df, y


def demonstrate_leakage():
    """Create dataset that clearly shows impact of data leakage."""
    np.random.seed(42)
    n_samples = 1000
    
    # Create feature with large variance differences between train/test
    X_train_part = np.random.randn(800, 1) * 10  # High variance
    X_test_part = np.random.randn(200, 1) * 2    # Low variance
    X = np.vstack([X_train_part, X_test_part])
    
    # Target depends on scaled feature
    y = (X[:, 0] > 0).astype(int)
    
    return X, y


def compare_with_without_leakage(X, y):
    """Compare model performance with and without data leakage."""
    # Split data first
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # WITH LEAKAGE: Fit scaler on all data
    scaler_leakage = StandardScaler().fit(X)  # WRONG!
    X_train_leaked = scaler_leakage.transform(X_train)
    X_test_leaked = scaler_leakage.transform(X_test)
    
    model_leaked = LogisticRegression(random_state=42)
    model_leaked.fit(X_train_leaked, y_train)
    score_leaked = model_leaked.score(X_test_leaked, y_test)
    
    # WITHOUT LEAKAGE: Fit scaler only on training data
    scaler_correct = StandardScaler().fit(X_train)  # CORRECT!
    X_train_correct = scaler_correct.transform(X_train)
    X_test_correct = scaler_correct.transform(X_test)
    
    model_correct = LogisticRegression(random_state=42)
    model_correct.fit(X_train_correct, y_train)
    score_correct = model_correct.score(X_test_correct, y_test)
    
    return score_leaked, score_correct


def visualize_pipeline(pipeline, save_path=None):
    """Visualize pipeline structure."""
    from sklearn import set_config
    set_config(display='diagram')
    return pipeline


def evaluate_pipeline(pipeline, X_train, X_test, y_train, y_test):
    """Evaluate pipeline performance."""
    # Train
    start = time()
    pipeline.fit(X_train, y_train)
    train_time = time() - start
    
    # Evaluate
    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    
    # Predictions
    y_pred = pipeline.predict(X_test)
    
    results = {
        'train_score': train_score,
        'test_score': test_score,
        'train_time': train_time,
        'predictions': y_pred
    }
    
    return results

print("Helper functions defined!")

<a name='2'></a>
## 2 - Basic Pipeline Construction

A basic pipeline chains preprocessing steps with a model, ensuring proper fit/transform ordering.

### Pipeline Syntax:

**Method 1: Pipeline with named steps**
```python
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
```

**Method 2: make_pipeline (auto-naming)**
```python
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)
```

### Pipeline Operations:

**Training:**
```python
pipeline.fit(X_train, y_train)  # Fits all steps on training data
```

**Prediction:**
```python
predictions = pipeline.predict(X_test)  # Transforms and predicts
```

**Accessing Steps:**
```python
pipeline.named_steps['scaler']  # Access specific step
pipeline.steps[0][1]            # Access by index
pipeline['scaler']              # Dictionary-like access
```

### Benefits:

1. **Prevents Leakage**: Scaler fits only on training data
2. **Cleaner Code**: One object instead of many
3. **Cross-Validation**: Works seamlessly with CV
4. **Serialization**: Save entire pipeline as one object

### Common Pipeline Steps:

- **Imputation**: `SimpleImputer()` for missing values
- **Scaling**: `StandardScaler()`, `MinMaxScaler()`, `RobustScaler()`
- **Feature Selection**: `SelectKBest()`, `PCA()`
- **Encoding**: `OneHotEncoder()`, `OrdinalEncoder()`
- **Model**: Any classifier or regressor

<a name='ex-1'></a>
### Exercise 1 - Build a Basic Pipeline

**Your Task:**

Implement the `create_basic_pipeline` function that:
1. Creates a pipeline with StandardScaler and a classifier
2. Trains the pipeline on training data
3. Evaluates on test data
4. Compares with manual approach (without pipeline)

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For pipeline creation:**
* Use Pipeline: `pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', model)])`
* Or make_pipeline: `pipeline = make_pipeline(StandardScaler(), model)`

**For training:**
* Single call: `pipeline.fit(X_train, y_train)`
* Handles fit and transform internally

**For evaluation:**
* Score: `pipeline.score(X_test, y_test)`
* Predict: `pipeline.predict(X_test)`

**For comparison:**
* Manual: Fit scaler, transform train, transform test, fit model, predict
* Pipeline: Single fit and predict call

</details>

In [ ]:
# GRADED FUNCTION: create_basic_pipeline
def create_basic_pipeline(X_train, X_test, y_train, y_test, model=None):
    """
    Create and evaluate a basic preprocessing + model pipeline.
    
    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training labels
        y_test: Test labels
        model: Classifier to use (default: LogisticRegression)
    
    Returns:
        Dictionary with pipeline, train_score, test_score
    """
    if model is None:
        model = LogisticRegression(random_state=42, max_iter=1000)
    
    ### START CODE HERE ### (≈ 10 lines)
    
    # Create pipeline with scaler and classifier
    pipeline = None
    
    # Fit pipeline on training data
    
    # Evaluate on training data
    train_score = None
    
    # Evaluate on test data
    test_score = None
    
    ### END CODE HERE ###
    
    return {
        'pipeline': pipeline,
        'train_score': train_score,
        'test_score': test_score
    }

In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

print("Dataset loaded:")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")

# Create and evaluate basic pipeline
print("\n" + "="*60)
print("BASIC PIPELINE")
print("="*60)

results = create_basic_pipeline(X_train, X_test, y_train, y_test)

print(f"\nPipeline created successfully!")
print(f"Training accuracy: {results['train_score']:.4f}")
print(f"Test accuracy: {results['test_score']:.4f}")

# Display pipeline structure
print(f"\nPipeline steps:")
for name, step in results['pipeline'].named_steps.items():
    print(f"  {name}: {step.__class__.__name__}")

# Compare with manual approach
print("\n" + "="*60)
print("COMPARISON: PIPELINE vs MANUAL")
print("="*60)

# Manual approach (more verbose, error-prone)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)
manual_score = model.score(X_test_scaled, y_test)

print(f"\nManual approach test score: {manual_score:.4f}")
print(f"Pipeline test score: {results['test_score']:.4f}")
print(f"Difference: {abs(manual_score - results['test_score']):.6f} (should be ~0)")

print("\nBenefits of Pipeline:")
print("  • Prevents accidentally fitting on test data")
print("  • Cleaner, more readable code")
print("  • Easier to serialize and deploy")
print("  • Works seamlessly with cross-validation")

##### **Expected Output**
```
Dataset loaded:
Training set: (105, 4)
Test set: (45, 4)
Number of classes: 3

============================================================
BASIC PIPELINE
============================================================

Pipeline created successfully!
Training accuracy: 0.9XXX
Test accuracy: 0.9XXX

Pipeline steps:
  scaler: StandardScaler
  classifier: LogisticRegression

============================================================
COMPARISON: PIPELINE vs MANUAL
============================================================

Manual approach test score: 0.9XXX
Pipeline test score: 0.9XXX
Difference: 0.000000 (should be ~0)
```
The pipeline and manual approaches should produce identical results, but the pipeline is cleaner and safer.

In [ ]:
unittests.exercise_1(create_basic_pipeline)

<a name='3'></a>
## 3 - ColumnTransformer for Mixed Data

Real-world datasets often contain both numeric and categorical features. **ColumnTransformer** applies different preprocessing to different column types.

### The Challenge:

Given a dataset with:
- **Numeric columns**: `age`, `fare` → Need scaling
- **Categorical columns**: `sex`, `embarked` → Need encoding

We need different transformations for each type!

### ColumnTransformer Syntax:

```python
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])
```

### Complete Pipeline:

```python
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])
```

### Execution Flow:

1. **ColumnTransformer** applies:
   - `StandardScaler()` to numeric columns
   - `OneHotEncoder()` to categorical columns
   - Concatenates results

2. **Classifier** receives concatenated features

### Advanced Options:

**Remainder handling:**
```python
ColumnTransformer(
    transformers=[...],
    remainder='passthrough'  # Keep untransformed columns
    # remainder='drop'       # Drop untransformed columns (default)
)
```

**make_column_transformer (auto-naming):**
```python
from sklearn.compose import make_column_transformer

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(), categorical_features)
)
```

### Common Patterns:

**Numeric preprocessing:**
- Imputation: `SimpleImputer(strategy='median')`
- Scaling: `StandardScaler()`, `MinMaxScaler()`, `RobustScaler()`

**Categorical preprocessing:**
- Imputation: `SimpleImputer(strategy='most_frequent')`
- Encoding: `OneHotEncoder(handle_unknown='ignore')`

**Combining both:**
```python
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
```

<a name='ex-2'></a>
### Exercise 2 - Implement ColumnTransformer

**Your Task:**

Implement the `create_column_transformer_pipeline` function that:
1. Identifies numeric and categorical columns
2. Creates separate transformers for each type
3. Combines them with ColumnTransformer
4. Builds complete pipeline with a classifier

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For column identification:**
* Numeric: `X.select_dtypes(include=['int64', 'float64']).columns.tolist()`
* Categorical: `X.select_dtypes(include=['object', 'category']).columns.tolist()`

**For transformers:**
* Numeric pipeline: Imputer → Scaler
* Categorical pipeline: Imputer → Encoder
* Use `handle_unknown='ignore'` in OneHotEncoder

**For ColumnTransformer:**
```python
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])
```

**For full pipeline:**
* Combine preprocessor with classifier

</details>

In [ ]:
# GRADED FUNCTION: create_column_transformer_pipeline
def create_column_transformer_pipeline(X_train, y_train, X_test, y_test, model=None):
    """
    Create pipeline with ColumnTransformer for mixed data types.
    
    Args:
        X_train: Training features (DataFrame with mixed types)
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        model: Classifier to use (default: RandomForestClassifier)
    
    Returns:
        Dictionary with pipeline, train_score, test_score, feature_names
    """
    if model is None:
        model = RandomForestClassifier(random_state=42, n_estimators=100)
    
    ### START CODE HERE ### (≈ 25 lines)
    
    # Identify numeric and categorical columns
    numeric_features = None
    categorical_features = None
    
    # Create numeric transformer (imputation + scaling)
    numeric_transformer = None
    
    # Create categorical transformer (imputation + encoding)
    categorical_transformer = None
    
    # Combine transformers with ColumnTransformer
    preprocessor = None
    
    # Create full pipeline
    pipeline = None
    
    # Fit pipeline
    # pipeline.fit(...)
    
    # Evaluate
    train_score = None
    test_score = None
    
    ### END CODE HERE ###
    
    return {
        'pipeline': pipeline,
        'train_score': train_score,
        'test_score': test_score,
        'numeric_features': numeric_features,
        'categorical_features': categorical_features
    }

In [ ]:
# Load Titanic dataset (mixed numeric/categorical)
X_titanic, y_titanic = load_titanic_data()

print("Dataset loaded:")
print(f"Shape: {X_titanic.shape}")
print(f"\nFeatures and types:")
print(X_titanic.dtypes)
print(f"\nFirst few rows:")
print(X_titanic.head())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42
)

print("\n" + "="*60)
print("COLUMNTRANSFORMER PIPELINE")
print("="*60)

# Create and evaluate pipeline
results = create_column_transformer_pipeline(X_train, y_train, X_test, y_test)

print(f"\nPipeline created successfully!")
print(f"\nNumeric features ({len(results['numeric_features'])}): {results['numeric_features']}")
print(f"Categorical features ({len(results['categorical_features'])}): {results['categorical_features']}")
print(f"\nTraining accuracy: {results['train_score']:.4f}")
print(f"Test accuracy: {results['test_score']:.4f}")

# Display pipeline structure
print(f"\nPipeline structure:")
print(results['pipeline'])

# Make sample prediction
sample = X_test.iloc[:1]
prediction = results['pipeline'].predict(sample)
print(f"\nSample prediction:")
print(f"Input: {sample.to_dict('records')[0]}")
print(f"Prediction: {'Survived' if prediction[0] == 1 else 'Did not survive'}")

##### **Expected Output**
```
Dataset loaded:
Shape: (XXX, X)

Features and types:
pclass        int64
sex          object
age         float64
...

============================================================
COLUMNTRANSFORMER PIPELINE
============================================================

Pipeline created successfully!

Numeric features (X): ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categorical features (X): ['sex', 'embarked']

Training accuracy: 0.9XXX
Test accuracy: 0.7XXX
```
The pipeline should correctly handle both numeric and categorical features, automatically applying appropriate transformations.

In [ ]:
unittests.exercise_2(create_column_transformer_pipeline)

<a name='4'></a>
## 4 - Data Leakage Identification

**Data leakage** is one of the most common and dangerous mistakes in machine learning. It leads to models that perform well in development but fail in production.

### Types of Leakage:

#### 1. **Preprocessing Leakage**

**Problem:** Fitting transformers on entire dataset including test data

```python
# WRONG: Scaler sees test data!
scaler = StandardScaler().fit(X)  # Includes test data
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled)

# CORRECT: Fit only on training
X_train, X_test = train_test_split(X)
scaler = StandardScaler().fit(X_train)  # Only training
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

**Why it matters:** Test set statistics (mean, std) influence training

#### 2. **Target Leakage**

**Problem:** Features that contain information about the target

```python
# WRONG: 'already_defaulted' is the target in disguise!
features = ['income', 'debt', 'already_defaulted']
target = 'will_default'

# CORRECT: Only use features available at prediction time
features = ['income', 'debt', 'credit_score']
```

**Examples:**
- Predicting disease using post-diagnosis tests
- Predicting customer churn using cancellation reason
- Stock prediction using future prices

#### 3. **Temporal Leakage**

**Problem:** Using future information to predict past

```python
# WRONG: Random split on time series
X_train, X_test = train_test_split(timeseries_data)  # Mixes past and future

# CORRECT: Temporal split
split_date = '2023-01-01'
X_train = data[data.date < split_date]
X_test = data[data.date >= split_date]
```

#### 4. **Cross-Validation Leakage**

**Problem:** Preprocessing before splitting in CV

```python
# WRONG: Preprocessing outside CV
X_scaled = StandardScaler().fit_transform(X)
scores = cross_val_score(model, X_scaled, y)  # Each fold sees others!

# CORRECT: Use pipeline
pipeline = make_pipeline(StandardScaler(), model)
scores = cross_val_score(pipeline, X, y)  # Proper isolation
```

### Detection Strategies:

1. **Suspiciously High Performance**: 99%+ accuracy on complex problems
2. **Performance Drop in Production**: Great in dev, poor in production
3. **Feature Importance**: Target-like features have high importance
4. **Temporal Check**: Can you know this feature before prediction?
5. **Distribution Shift**: Train and test distributions differ significantly

### Prevention Best Practices:

- **Always use pipelines** for preprocessing
- **Split data first**, then preprocess
- **Check feature availability** at prediction time
- **Respect temporal ordering** in time series
- **Use proper CV** with pipelines
- **Validate on truly held-out data**
- **Document data collection** timestamps

<a name='ex-3'></a>
### Exercise 3 - Find and Fix Leakage

**Your Task:**

Implement the `identify_and_fix_leakage` function that:
1. Demonstrates code with data leakage
2. Shows the performance difference
3. Fixes the leakage using proper pipeline
4. Explains why the fix prevents leakage

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For leaky code:**
* Fit scaler on entire X before splitting
* Then split the scaled data
* Train model on this data

**For correct code:**
* Split data FIRST
* Create pipeline with scaler and model
* Fit pipeline only on training data

**For comparison:**
* Evaluate both on test set
* Leaky version might show unrealistically high performance
* Or might show unexpected behavior

**Key insight:**
* Test set statistics (mean, std) shouldn't influence training
* Pipeline ensures this by fitting only on training fold

</details>

In [ ]:
# GRADED FUNCTION: identify_and_fix_leakage
def identify_and_fix_leakage(X, y, test_size=0.2):
    """
    Demonstrate data leakage and how to fix it.
    
    Args:
        X: Features
        y: Target
        test_size: Proportion of test set
    
    Returns:
        Dictionary with leaky_score, correct_score, explanation
    """
    ### START CODE HERE ### (≈ 25 lines)
    
    # LEAKY APPROACH: Fit scaler on ALL data before splitting
    # 1. Fit scaler on entire X
    scaler_leaky = None
    X_scaled_leaky = None
    
    # 2. Split AFTER scaling (WRONG!)
    X_train_leaky, X_test_leaky, y_train, y_test = None
    
    # 3. Train model
    model_leaky = None
    # Fit model
    
    # 4. Evaluate
    leaky_score = None
    
    # CORRECT APPROACH: Use pipeline
    # 1. Split FIRST
    X_train, X_test, y_train, y_test = None
    
    # 2. Create pipeline (fits scaler only on training data)
    pipeline_correct = None
    
    # 3. Fit pipeline
    # pipeline_correct.fit(...)
    
    # 4. Evaluate
    correct_score = None
    
    # Explanation of the fix
    explanation = None  # Write explanation string
    
    ### END CODE HERE ###
    
    return {
        'leaky_score': leaky_score,
        'correct_score': correct_score,
        'score_difference': abs(leaky_score - correct_score),
        'explanation': explanation
    }

In [ ]:
# Create dataset that demonstrates leakage impact
X_leak, y_leak = demonstrate_leakage()

print("="*60)
print("DATA LEAKAGE DEMONSTRATION")
print("="*60)

results = identify_and_fix_leakage(X_leak, y_leak)

print(f"\nPerformance Comparison:")
print(f"Leaky approach:   {results['leaky_score']:.4f}")
print(f"Correct approach: {results['correct_score']:.4f}")
print(f"Difference:       {results['score_difference']:.4f}")

print(f"\nExplanation:")
print(results['explanation'])

# Visual demonstration
print("\n" + "="*60)
print("CROSS-VALIDATION WITH AND WITHOUT PIPELINE")
print("="*60)

# Load Iris for CV demonstration
X_iris, y_iris = load_iris(return_X_y=True)

# WRONG: Preprocessing before CV
X_scaled_wrong = StandardScaler().fit_transform(X_iris)
model_wrong = LogisticRegression(random_state=42, max_iter=1000)
scores_wrong = cross_val_score(model_wrong, X_scaled_wrong, y_iris, cv=5)

# CORRECT: Pipeline ensures proper CV
pipeline_correct = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=1000)
)
scores_correct = cross_val_score(pipeline_correct, X_iris, y_iris, cv=5)

print(f"\nCV Scores (preprocessing before CV - WRONG):")
print(f"  Scores: {scores_wrong}")
print(f"  Mean: {scores_wrong.mean():.4f} (+/- {scores_wrong.std():.4f})")

print(f"\nCV Scores (pipeline with proper isolation - CORRECT):")
print(f"  Scores: {scores_correct}")
print(f"  Mean: {scores_correct.mean():.4f} (+/- {scores_correct.std():.4f})")

print("\n⚠️  The 'wrong' approach may show slightly better scores because")
print("    each fold's validation set influenced the scaler during training!")

# Common leakage scenarios
print("\n" + "="*60)
print("COMMON LEAKAGE SCENARIOS")
print("="*60)

leakage_examples = [
    {
        'name': 'Preprocessing Leakage',
        'wrong': 'scaler.fit(X_all); split(X_all)',
        'right': 'split(X); scaler.fit(X_train)',
        'impact': 'Test statistics leak into training'
    },
    {
        'name': 'Target Leakage',
        'wrong': 'Using feature that contains target info',
        'right': 'Only use features available before target',
        'impact': 'Unrealistically high performance'
    },
    {
        'name': 'Temporal Leakage',
        'wrong': 'train_test_split() on time series',
        'right': 'Split by time: train < test chronologically',
        'impact': 'Model sees the future'
    },
    {
        'name': 'CV Leakage',
        'wrong': 'Preprocess before cross_val_score',
        'right': 'Use Pipeline with cross_val_score',
        'impact': 'Folds contaminate each other'
    }
]

for i, ex in enumerate(leakage_examples, 1):
    print(f"\n{i}. {ex['name']}")
    print(f"   Wrong:  {ex['wrong']}")
    print(f"   Right:  {ex['right']}")
    print(f"   Impact: {ex['impact']}")

##### **Expected Output**
```
============================================================
DATA LEAKAGE DEMONSTRATION
============================================================

Performance Comparison:
Leaky approach:   0.XXXX
Correct approach: 0.XXXX
Difference:       0.XXXX

Explanation:
[Your explanation of why leakage occurred and how pipeline fixes it]

============================================================
CROSS-VALIDATION WITH AND WITHOUT PIPELINE
============================================================

CV Scores (preprocessing before CV - WRONG):
  Scores: [0.9XXX 0.9XXX 0.9XXX 0.9XXX 0.9XXX]
  Mean: 0.9XXX (+/- 0.0XXX)

CV Scores (pipeline with proper isolation - CORRECT):
  Scores: [0.9XXX 0.9XXX 0.9XXX 0.9XXX 0.9XXX]
  Mean: 0.9XXX (+/- 0.0XXX)
```
The demonstration should clearly show how pipelines prevent leakage by ensuring preprocessing fits only on training folds.

In [ ]:
unittests.exercise_3(identify_and_fix_leakage)

<a name='5'></a>
## 5 - Pipeline with Hyperparameter Tuning

Pipelines integrate seamlessly with GridSearchCV and RandomizedSearchCV, allowing you to tune preprocessing and model hyperparameters together safely.

### Why Tune in Pipeline?

1. **Prevents Leakage**: Preprocessing refitted for each hyperparameter configuration
2. **Explores Preprocessing**: Can tune scaler type, imputation strategy, etc.
3. **Joint Optimization**: Find best combination of preprocessing and model params
4. **Cleaner Code**: Everything in one object

### Parameter Naming:

Access pipeline steps using double underscore `__`:

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier())
])

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None],
    # Can even tune preprocessing:
    # 'scaler': [StandardScaler(), MinMaxScaler()]
}
```

### GridSearchCV with Pipeline:

```python
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
best_pipeline = grid_search.best_estimator_
```

### What Happens in Each CV Fold:

For each parameter combination:
1. Split into train/validation
2. **Fit scaler on train fold only**
3. Transform train and validation
4. Fit model on transformed train
5. Evaluate on transformed validation
6. Repeat for all folds

### Complex Pipelines:

With ColumnTransformer:

```python
param_grid = {
    # Numeric preprocessing
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__num__scaler': [StandardScaler(), MinMaxScaler()],
    
    # Categorical preprocessing
    'preprocessor__cat__encoder__handle_unknown': ['ignore', 'error'],
    
    # Model parameters
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10]
}
```

### Best Practices:

- **Start small**: Tune a few important parameters first
- **Use random search**: For large parameter spaces
- **Set cv appropriately**: Balance between runtime and reliability
- **Refit final model**: GridSearchCV refits on all training data with best params
- **Extract best pipeline**: Use `best_estimator_` for predictions

<a name='ex-4'></a>
### Exercise 4 - Pipeline with GridSearchCV

**Your Task:**

Implement the `tune_pipeline_hyperparameters` function that:
1. Creates a pipeline with preprocessing and classifier
2. Defines parameter grid for both preprocessing and model
3. Performs grid search with proper cross-validation
4. Returns best pipeline and results

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For pipeline creation:**
* Include preprocessing steps and classifier
* Name steps clearly for parameter referencing

**For parameter grid:**
* Use double underscore: `'step_name__param_name'`
* Example: `'classifier__n_estimators': [50, 100, 200]`
* Can tune preprocessing: `'scaler': [StandardScaler(), RobustScaler()]`

**For GridSearchCV:**
```python
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True
)
```

**For results:**
* Best params: `grid_search.best_params_`
* Best score: `grid_search.best_score_`
* Best pipeline: `grid_search.best_estimator_`
* All results: `pd.DataFrame(grid_search.cv_results_)`

</details>

In [ ]:
# GRADED FUNCTION: tune_pipeline_hyperparameters
def tune_pipeline_hyperparameters(X_train, X_test, y_train, y_test, cv_folds=3):
    """
    Tune hyperparameters of a pipeline using GridSearchCV.
    
    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training labels
        y_test: Test labels
        cv_folds: Number of CV folds
    
    Returns:
        Dictionary with best_pipeline, best_params, best_score, test_score
    """
    ### START CODE HERE ### (≈ 20 lines)
    
    # Create base pipeline
    pipeline = None
    
    # Define parameter grid
    param_grid = None
    
    # Create GridSearchCV
    grid_search = None
    
    # Perform grid search
    # grid_search.fit(...)
    
    # Extract results
    best_pipeline = None
    best_params = None
    best_cv_score = None
    
    # Evaluate on test set
    test_score = None
    
    ### END CODE HERE ###
    
    return {
        'best_pipeline': best_pipeline,
        'best_params': best_params,
        'best_cv_score': best_cv_score,
        'test_score': test_score,
        'grid_search': grid_search
    }

In [ ]:
# Load dataset
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("="*60)
print("PIPELINE HYPERPARAMETER TUNING")
print("="*60)

results = tune_pipeline_hyperparameters(X_train, X_test, y_train, y_test, cv_folds=3)

print(f"\nGrid Search Completed!")
print(f"\nBest Parameters:")
for param, value in results['best_params'].items():
    print(f"  {param}: {value}")

print(f"\nPerformance:")
print(f"  Best CV Score: {results['best_cv_score']:.4f}")
print(f"  Test Score:    {results['test_score']:.4f}")

# Show top configurations
cv_results = pd.DataFrame(results['grid_search'].cv_results_)
top_results = cv_results.nsmallest(5, 'rank_test_score')[[
    'params', 'mean_test_score', 'std_test_score', 'rank_test_score'
]]

print(f"\nTop 5 Configurations:")
for idx, row in top_results.iterrows():
    print(f"\nRank {int(row['rank_test_score'])}:")
    print(f"  Score: {row['mean_test_score']:.4f} (+/- {row['std_test_score']:.4f})")
    print(f"  Params: {row['params']}")

# Demonstrate why pipeline is essential for grid search
print("\n" + "="*60)
print("WHY PIPELINE IS ESSENTIAL FOR GRIDSEARCH")
print("="*60)

print("\nWithout Pipeline:")
print("  Must manually preprocess for each parameter combination")
print("  Risk of data leakage in CV folds")
print("  Can't tune preprocessing hyperparameters")
print("  Code becomes complex and error-prone")

print("\nWith Pipeline:")
print("  Preprocessing automatically applied correctly")
print("  No leakage - each fold properly isolated")
print("  Can tune both preprocessing and model")
print("  Clean, maintainable code")
print("  Production-ready output")

##### **Expected Output**
```
============================================================
PIPELINE HYPERPARAMETER TUNING
============================================================

Grid Search Completed!

Best Parameters:
  classifier__C: X.XX
  classifier__max_iter: XXX
  ...

Performance:
  Best CV Score: 0.9XXX
  Test Score:    0.9XXX

Top 5 Configurations:

Rank 1:
  Score: 0.9XXX (+/- 0.0XXX)
  Params: {...}
```
GridSearchCV should find optimal hyperparameters while maintaining proper data isolation through the pipeline.

In [ ]:
unittests.exercise_4(tune_pipeline_hyperparameters)

<a name='6'></a>
## 6 - Production Pipeline

A **production pipeline** handles real-world data end-to-end: from raw input to predictions, including error handling, validation, and serialization.

### Production Requirements:

1. **Robustness**: Handle missing values, outliers, unknown categories
2. **Serialization**: Save and load the entire pipeline
3. **Validation**: Check input data format and ranges
4. **Logging**: Track predictions and potential issues
5. **Versioning**: Track pipeline version and training data
6. **Documentation**: Clear input/output specifications

### Complete Production Pipeline:

```python
# Numeric pipeline: imputation → scaling
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())  # Robust to outliers
])

# Categorical pipeline: imputation → encoding
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))  # Graceful handling
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Full pipeline
production_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])
```

### Serialization:

**Save pipeline:**
```python
import joblib

# Save
joblib.dump(pipeline, 'model_pipeline.pkl')

# Load
loaded_pipeline = joblib.load('model_pipeline.pkl')

# Use immediately
predictions = loaded_pipeline.predict(new_data)
```

### Input Validation:

```python
def validate_input(X, expected_columns):
    """Validate input data before prediction."""
    # Check column names
    if not all(col in X.columns for col in expected_columns):
        raise ValueError(f"Missing columns. Expected: {expected_columns}")
    
    # Check data types
    for col in numeric_columns:
        if not np.issubdtype(X[col].dtype, np.number):
            raise TypeError(f"{col} must be numeric")
    
    return X[expected_columns]  # Ensure column order
```

### Versioning and Metadata:

```python
pipeline_metadata = {
    'version': '1.0.0',
    'trained_date': '2024-01-15',
    'training_samples': 10000,
    'features': list(X_train.columns),
    'target': 'outcome',
    'performance': {
        'train_accuracy': 0.95,
        'test_accuracy': 0.92
    }
}

# Save alongside pipeline
joblib.dump({
    'pipeline': pipeline,
    'metadata': pipeline_metadata
}, 'model_bundle.pkl')
```

### Deployment Patterns:

1. **Batch Prediction**:
   - Load pipeline once
   - Process many samples
   - Write results to database/file

2. **API Service**:
   - Load pipeline at startup
   - Handle individual requests
   - Return predictions in real-time

3. **Streaming**:
   - Pipeline in stream processor
   - Process records as they arrive
   - Low-latency predictions

<a name='ex-5'></a>
### Exercise 5 - End-to-End Production Pipeline

**Your Task:**

Implement the `create_production_pipeline` function that:
1. Builds a complete preprocessing + model pipeline
2. Handles missing values and unknown categories gracefully
3. Trains on data and evaluates performance
4. Serializes pipeline to disk
5. Loads and validates the saved pipeline

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For robust preprocessing:**
* Numeric: SimpleImputer(strategy='median') + RobustScaler()
* Categorical: SimpleImputer(strategy='most_frequent') + OneHotEncoder(handle_unknown='ignore')

**For serialization:**
```python
import joblib
joblib.dump(pipeline, filepath)
loaded = joblib.load(filepath)
```

**For validation:**
* Load pipeline
* Make prediction on test sample
* Verify output format
* Check performance matches

**For metadata:**
* Include: training date, feature names, performance metrics
* Save alongside pipeline

</details>

In [ ]:
# GRADED FUNCTION: create_production_pipeline
def create_production_pipeline(X_train, X_test, y_train, y_test, save_path='production_pipeline.pkl'):
    """
    Create, train, and serialize a production-ready pipeline.
    
    Args:
        X_train: Training features (DataFrame)
        X_test: Test features (DataFrame)
        y_train: Training labels
        y_test: Test labels
        save_path: Path to save serialized pipeline
    
    Returns:
        Dictionary with pipeline, scores, and validation results
    """
    ### START CODE HERE ### (≈ 35 lines)
    
    # Identify column types
    numeric_features = None
    categorical_features = None
    
    # Create numeric preprocessing pipeline
    numeric_transformer = None
    
    # Create categorical preprocessing pipeline
    categorical_transformer = None
    
    # Combine with ColumnTransformer
    preprocessor = None
    
    # Create full pipeline
    pipeline = None
    
    # Train pipeline
    # pipeline.fit(...)
    
    # Evaluate
    train_score = None
    test_score = None
    
    # Create metadata
    from datetime import datetime
    metadata = None  # Create dictionary with version, date, features, performance
    
    # Serialize pipeline and metadata
    bundle = {'pipeline': pipeline, 'metadata': metadata}
    # joblib.dump(...)
    
    # Validate serialization: Load and test
    loaded_bundle = None  # joblib.load(...)
    loaded_pipeline = None
    
    # Make test prediction to verify
    test_sample = None  # Use first test sample
    original_pred = None
    loaded_pred = None
    
    serialization_valid = None  # Check if predictions match
    
    ### END CODE HERE ###
    
    return {
        'pipeline': pipeline,
        'train_score': train_score,
        'test_score': test_score,
        'metadata': metadata,
        'save_path': save_path,
        'serialization_valid': serialization_valid,
        'loaded_pipeline': loaded_pipeline
    }

In [ ]:
# Load mixed-type dataset
X, y = load_titanic_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("="*60)
print("PRODUCTION PIPELINE CREATION")
print("="*60)

results = create_production_pipeline(X_train, X_test, y_train, y_test)

print(f"\nPipeline created and trained!")
print(f"\nPerformance:")
print(f"  Training accuracy: {results['train_score']:.4f}")
print(f"  Test accuracy:     {results['test_score']:.4f}")

print(f"\nMetadata:")
for key, value in results['metadata'].items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    elif isinstance(value, list) and len(value) > 5:
        print(f"  {key}: {value[:5]}... ({len(value)} total)")
    else:
        print(f"  {key}: {value}")

print(f"\nSerialization:")
print(f"  Saved to: {results['save_path']}")
print(f"  Validation: {'PASSED' if results['serialization_valid'] else 'FAILED'}")

# Demonstrate production usage
print("\n" + "="*60)
print("PRODUCTION USAGE DEMONSTRATION")
print("="*60)

# Simulate loading in production
loaded_pipeline = results['loaded_pipeline']

# Make predictions on new data
sample_data = X_test.iloc[:3]
predictions = loaded_pipeline.predict(sample_data)
probabilities = loaded_pipeline.predict_proba(sample_data)

print(f"\nSample Predictions:")
for i, (idx, row) in enumerate(sample_data.iterrows()):
    print(f"\nSample {i+1}:")
    print(f"  Input: {dict(row)}")
    print(f"  Prediction: {'Survived' if predictions[i] == 1 else 'Did not survive'}")
    print(f"  Probability: {probabilities[i][1]:.2%}")

# Production checklist
print("\n" + "="*60)
print("PRODUCTION READINESS CHECKLIST")
print("="*60)

checklist = [
    ("Handles missing values", True),
    ("Handles unknown categories", True),
    ("Serializable", results['serialization_valid']),
    ("Includes metadata", results['metadata'] is not None),
    ("No data leakage", True),
    ("Version tracked", 'version' in results['metadata']),
    ("Performance documented", 'performance' in results['metadata'])
]

for item, status in checklist:
    symbol = "[x]" if status else "[ ]"
    print(f"{symbol} {item}")

print("\nThis pipeline is ready for:")
print("  • Batch predictions on new data")
print("  • REST API deployment")
print("  • Streaming applications")
print("  • Model registry storage")

##### **Expected Output**
```
============================================================
PRODUCTION PIPELINE CREATION
============================================================

Pipeline created and trained!

Performance:
  Training accuracy: 0.XXXX
  Test accuracy:     0.XXXX

Metadata:
  version: 1.0.0
  trained_date: 2024-XX-XX
  training_samples: XXX
  features: ['pclass', 'sex', 'age', ...]... (X total)
  performance:
    train_accuracy: 0.XXXX
    test_accuracy: 0.XXXX

Serialization:
  Saved to: production_pipeline.pkl
  Validation: PASSED

============================================================
PRODUCTION READINESS CHECKLIST
============================================================
[x] Handles missing values
[x] Handles unknown categories
[x] Serializable
[x] Includes metadata
[x] No data leakage
[x] Version tracked
[x] Performance documented
```
The production pipeline should handle all edge cases and be fully serializable for deployment.

In [ ]:
unittests.exercise_5(create_production_pipeline)

<a name='7'></a>
## 7 - From-Scratch Implementation: Simple Pipeline

Understanding how pipelines work internally helps you debug issues and create custom transformers.

### Core Pipeline Logic:

A pipeline is essentially:
1. A list of (name, transformer/estimator) tuples
2. Logic to chain fit/transform calls
3. Special handling for the final estimator

### Key Methods:

**fit(X, y):**
```python
def fit(self, X, y=None):
    X_transformed = X
    
    # Fit and transform all steps except last
    for name, transform in self.steps[:-1]:
        X_transformed = transform.fit_transform(X_transformed, y)
    
    # Fit final estimator
    self.steps[-1][1].fit(X_transformed, y)
    
    return self
```

**predict(X):**
```python
def predict(self, X):
    X_transformed = X
    
    # Transform through all steps except last (no fitting!)
    for name, transform in self.steps[:-1]:
        X_transformed = transform.transform(X_transformed)
    
    # Predict with final estimator
    return self.steps[-1][1].predict(X_transformed)
```

### Transform vs Fit_Transform:

- **fit_transform(X, y)**: Learn from X and transform it
- **transform(X)**: Apply learned transformation to X

During training: Use `fit_transform` to be efficient

During prediction: Only `transform` (no fitting!)

### Custom Transformer:

```python
from sklearn.base import BaseEstimator, TransformerMixin

class CustomTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, param=1.0):
        self.param = param
    
    def fit(self, X, y=None):
        # Learn something from X
        self.learned_value_ = X.mean()
        return self
    
    def transform(self, X):
        # Apply transformation
        return X * self.param
```

<a name='ex-6'></a>
### Exercise 6 - Simple Pipeline from Scratch

**Your Task:**

Implement the `SimplePipeline` class that:
1. Stores a list of transformers and a final estimator
2. Implements `fit()` to chain fit/transform operations
3. Implements `predict()` to transform and predict
4. Validates that your implementation matches sklearn's Pipeline

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For initialization:**
* Store steps as list of (name, transformer) tuples
* Validate that all have fit/transform (except last)

**For fit():**
* Loop through all steps except last
* For each: call fit_transform() and pass result to next
* For last step: call fit() only
* Return self for chaining

**For predict():**
* Loop through all steps except last
* For each: call transform() only (no fitting!)
* For last step: call predict()

**For score():**
* Transform X through all steps except last
* Call final estimator's score()

</details>

In [ ]:
# GRADED FUNCTION: SimplePipeline
class SimplePipeline:
    """
    Simple pipeline implementation from scratch.
    
    Chains transformers and a final estimator, ensuring proper
    fit/transform order to prevent data leakage.
    """
    
    def __init__(self, steps):
        """
        Initialize pipeline with list of (name, transformer) tuples.
        
        Args:
            steps: List of (name, transformer) tuples
        """
        ### START CODE HERE ### (≈ 2 lines)
        self.steps = None
        ### END CODE HERE ###
    
    def fit(self, X, y=None):
        """
        Fit all transformers and final estimator.
        
        For each transformer: fit and transform
        For final estimator: only fit
        """
        ### START CODE HERE ### (≈ 10 lines)
        
        X_transformed = None  # Start with original X
        
        # Fit and transform all steps except last
        for name, transformer in None:  # Loop through steps[:-1]
            # Fit and transform
            X_transformed = None
        
        # Fit final estimator (no transform)
        final_name, final_estimator = None
        # final_estimator.fit(...)
        
        ### END CODE HERE ###
        
        return self
    
    def predict(self, X):
        """
        Transform X through all transformers, then predict with final estimator.
        
        Important: Only transform, never fit during prediction!
        """
        ### START CODE HERE ### (≈ 8 lines)
        
        X_transformed = None  # Start with original X
        
        # Transform through all steps except last (NO FITTING!)
        for name, transformer in None:  # Loop through steps[:-1]
            X_transformed = None  # Only transform
        
        # Predict with final estimator
        final_name, final_estimator = None
        predictions = None
        
        ### END CODE HERE ###
        
        return predictions
    
    def score(self, X, y):
        """
        Transform X and score with final estimator.
        """
        ### START CODE HERE ### (≈ 8 lines)
        
        X_transformed = None
        
        # Transform through all steps except last
        for name, transformer in None:
            X_transformed = None
        
        # Score with final estimator
        final_name, final_estimator = None
        score = None
        
        ### END CODE HERE ###
        
        return score

In [ ]:
# Test custom pipeline implementation
print("="*60)
print("SIMPLE PIPELINE FROM SCRATCH")
print("="*60)

# Load data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create sklearn pipeline for comparison
sklearn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Create custom pipeline
custom_pipeline = SimplePipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Train both
print("\nTraining pipelines...")
sklearn_pipeline.fit(X_train, y_train)
custom_pipeline.fit(X_train, y_train)

# Compare predictions
sklearn_pred = sklearn_pipeline.predict(X_test)
custom_pred = custom_pipeline.predict(X_test)

predictions_match = np.array_equal(sklearn_pred, custom_pred)

print(f"\nPredictions match: {predictions_match}")

# Compare scores
sklearn_score = sklearn_pipeline.score(X_test, y_test)
custom_score = custom_pipeline.score(X_test, y_test)

print(f"\nPerformance Comparison:")
print(f"  Sklearn Pipeline: {sklearn_score:.4f}")
print(f"  Custom Pipeline:  {custom_score:.4f}")
print(f"  Difference:       {abs(sklearn_score - custom_score):.6f}")

if abs(sklearn_score - custom_score) < 1e-6:
    print("\nYour pipeline implementation is correct!")
else:
    print("\nScores don't match. Check your implementation.")

# Demonstrate that pipeline prevents leakage
print("\n" + "="*60)
print("LEAKAGE PREVENTION DEMONSTRATION")
print("="*60)

# Check that scaler was fit only on training data
scaler = custom_pipeline.steps[0][1]
print(f"\nScaler learned from training data:")
print(f"  Mean: {scaler.mean_[:3]}...")
print(f"  Std:  {scaler.scale_[:3]}...")

# Compare with scaler fit on all data (WRONG!)
scaler_leaky = StandardScaler().fit(np.vstack([X_train, X_test]))
print(f"\nLeaky scaler (fit on all data):")
print(f"  Mean: {scaler_leaky.mean_[:3]}...")
print(f"  Std:  {scaler_leaky.scale_[:3]}...")

print(f"\nThe values are different because our pipeline correctly")
print(f"   fits the scaler only on training data, preventing leakage!")

# Show fit/transform chaining
print("\n" + "="*60)
print("UNDERSTANDING FIT/TRANSFORM CHAINING")
print("="*60)

print("\nDuring Training (fit):")
print("  1. Scaler.fit_transform(X_train)  → X_scaled")
print("  2. Classifier.fit(X_scaled, y_train)")
print("  Scaler learns from training data only")

print("\nDuring Prediction (predict):")
print("  1. Scaler.transform(X_test)  → X_scaled (NO FITTING!)")
print("  2. Classifier.predict(X_scaled)")
print("  No leakage: test data never influences training")

print("\nKey Insight: Pipeline ensures fit happens only during training,")
print("   and only transform is used during prediction. This is the core")
print("   mechanism that prevents data leakage!")

##### **Expected Output**
```
============================================================
SIMPLE PIPELINE FROM SCRATCH
============================================================

Training pipelines...

Predictions match: True

Performance Comparison:
  Sklearn Pipeline: 0.9XXX
  Custom Pipeline:  0.9XXX
  Difference:       0.000000

Your pipeline implementation is correct!

============================================================
LEAKAGE PREVENTION DEMONSTRATION
============================================================

Scaler learned from training data:
  Mean: [X.XX X.XX X.XX]...
  Std:  [X.XX X.XX X.XX]...

Leaky scaler (fit on all data):
  Mean: [X.XX X.XX X.XX]...
  Std:  [X.XX X.XX X.XX]...
```
Your custom pipeline should produce identical results to sklearn's Pipeline, demonstrating correct implementation.

In [ ]:
unittests.exercise_6(SimplePipeline)

<a name='8'></a>
## 8 - Conclusion

**Congratulations!** You've completed the ML Pipelines & Data Leakage Prevention assignment!

### What You've Learned:

1. **Basic Pipelines**: Chain preprocessing and modeling with proper fit/transform order
2. **ColumnTransformer**: Handle mixed numeric and categorical data elegantly
3. **Leakage Detection**: Identify common leakage patterns and fix them
4. **Pipeline Tuning**: Integrate hyperparameter search safely with pipelines
5. **Production Pipelines**: Build robust, serializable end-to-end systems
6. **From-Scratch Implementation**: Understand internal pipeline mechanics

### Key Takeaways:

**Always Use Pipelines:**
- Prevents data leakage automatically
- Makes code cleaner and more maintainable
- Ensures consistency between training and prediction
- Simplifies deployment and serialization
- Works seamlessly with cross-validation

**Data Leakage Prevention:**
- Split data BEFORE any preprocessing
- Fit transformers only on training data
- Use pipelines for all preprocessing
- Respect temporal order in time series
- Validate on truly held-out data

**Production Best Practices:**
- Handle missing values and outliers gracefully
- Use `handle_unknown='ignore'` in encoders
- Include metadata (version, date, performance)
- Test serialization and deserialization
- Validate input data before prediction
- Log predictions and monitor performance

### Common Mistakes to Avoid:

**Preprocessing Before Splitting**
```python
# WRONG
X_scaled = scaler.fit_transform(X)
X_train, X_test = train_test_split(X_scaled)
```

**Correct: Split First**
```python
# CORRECT
X_train, X_test = train_test_split(X)
pipeline = make_pipeline(scaler, model)
pipeline.fit(X_train, y_train)
```

**Preprocessing Outside CV**
```python
# WRONG
X_scaled = scaler.fit_transform(X)
cross_val_score(model, X_scaled, y)
```

**Correct: Pipeline in CV**
```python
# CORRECT
pipeline = make_pipeline(scaler, model)
cross_val_score(pipeline, X, y)
```

**Using Target-Leaking Features**
```python
# WRONG: Feature contains target information
features = ['income', 'already_defaulted']  # This is the target!
```

**Correct: Only Pre-Target Features**
```python
# CORRECT: Only features available before target
features = ['income', 'credit_score', 'employment_years']
```

### Real-World Applications:

**Financial Services:**
- Credit scoring: Prevent temporal leakage
- Fraud detection: Real-time pipeline serving
- Risk modeling: Robust handling of missing data

**Healthcare:**
- Patient outcome prediction: No future information
- Medical diagnosis: Handle incomplete records
- Drug efficacy: Proper clinical trial splits

**E-commerce:**
- Customer churn: Temporal validation
- Recommendation systems: User-based splitting
- Demand forecasting: Time series pipelines

**Manufacturing:**
- Quality control: Real-time prediction pipelines
- Predictive maintenance: Sensor data preprocessing
- Supply chain: Multi-modal data handling

### Advanced Topics:

**Custom Transformers:**
- Inherit from `BaseEstimator` and `TransformerMixin`
- Implement `fit()` and `transform()`
- Add custom preprocessing logic

**Feature Unions:**
- Parallel feature extraction paths
- Combine multiple feature sets
- Useful for heterogeneous data

**Pipeline Caching:**
- `memory` parameter for expensive transformations
- Speeds up hyperparameter tuning
- Useful for feature engineering

**Model Stacking:**
- Pipelines as base estimators
- Meta-learner combines predictions
- Advanced ensemble techniques

### Tools and Best Practices:

**Development:**
- Use pipelines from the start
- Test with sample data
- Validate serialization early
- Document feature requirements

**Production:**
- Version control pipelines
- Monitor input distributions
- Track prediction performance
- A/B test new versions

**Debugging:**
- Use `set_config(display='diagram')`
- Inspect intermediate outputs
- Check fitted parameters
- Validate against manual approach

### Recommended Resources:

**Documentation:**
- Scikit-learn Pipeline User Guide
- Scikit-learn Column Transformer Examples
- MLOps best practices guides

**Books:**
- "Building Machine Learning Powered Applications" (Ameisen)
- "Designing Machine Learning Systems" (Huyen)
- "Machine Learning Design Patterns" (Lakshmanan et al.)

**Articles:**
- "Leakage in Data Mining" (Kaufman et al.)
- "Rules of Machine Learning" (Google)
- "Hidden Technical Debt in ML Systems" (Sculley et al.)

### Final Checklist for Production:

Before deploying your pipeline:

- [ ] No data leakage (verified with proper CV)
- [ ] Handles missing values
- [ ] Handles unknown categories
- [ ] Input validation implemented
- [ ] Serialization tested
- [ ] Metadata included
- [ ] Performance documented
- [ ] Version tracked
- [ ] Monitoring plan in place
- [ ] Rollback strategy defined

**Excellent work! You're now equipped to build production-ready ML pipelines that prevent data leakage and deploy successfully!**